In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone per rules
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (files are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# helper to build compact summary for a dataframe
def compact_summary(df, name_col, sample_limit=50):
    # ensure timestamp converted to UTC then to Asia/Shanghai tz-aware datetimes
    ts = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # basic counts
    rows_count = len(df)
    unique_cmdb = df['cmdb_id'].unique()
    unique_cmdb_count = len(unique_cmdb)
    unique_cmdb_sample = ",".join(map(str, unique_cmdb[:sample_limit]))
    # name column unique values
    unique_names = df[name_col].unique()
    unique_names_count = len(unique_names)
    unique_names_sample = ",".join(map(str, unique_names[:sample_limit]))
    # min/max timestamps (as ISO strings)
    min_ts = ts.min()
    max_ts = ts.max()
    # Build a one-row DataFrame summary (compact)
    summary = pd.DataFrame([{
        'file': name_col.split('_')[0] if '_' in name_col else name_col,
        'rows_count': rows_count,
        'unique_cmdb_count': unique_cmdb_count,
        'unique_cmdb_sample': unique_cmdb_sample,
        'min_timestamp_asia_shanghai': min_ts.isoformat() if pd.notna(min_ts) else None,
        'max_timestamp_asia_shanghai': max_ts.isoformat() if pd.notna(max_ts) else None,
        f'unique_{name_col}_count': unique_names_count,
        f'unique_{name_col}_sample': unique_names_sample
    }])
    return summary, ts

# Metric summary and top5
metric_summary, metric_ts = compact_summary(metric_df, 'kpi_name', sample_limit=50)
# attach converted datetime column for readability in top5
metric_df['_dt_asia_shanghai'] = metric_ts
metric_top5 = metric_df.head(5).loc[:, ['_dt_asia_shanghai', 'timestamp', 'cmdb_id', 'kpi_name', 'value']]

# Trace summary and top5
trace_summary, trace_ts = compact_summary(trace_df, 'trace_name', sample_limit=50)
trace_df['_dt_asia_shanghai'] = trace_ts
trace_top5 = trace_df.head(5).loc[:, ['_dt_asia_shanghai', 'timestamp', 'cmdb_id', 'trace_name', 'value']]

# Display compact results (per rule: use variable names, separated by commas)
metric_summary, metric_top5, trace_summary, trace_top5

```
Out[1]:
```
```python
# Build a plain-English summary string from previous results and display it.
# Reuse existing variables: metric_summary, metric_top5, trace_summary, trace_top5

ms = metric_summary.iloc[0]
ts = trace_summary.iloc[0]

metric_top5_csv = metric_top5.to_csv(index=False)
trace_top5_csv = trace_top5.to_csv(index=False)

# keep sample lists compact (truncate if long)
def compact_sample(s, max_chars=200):
    if pd.isna(s):
        return ""
    return (s if len(s) <= max_chars else s[:max_chars].rstrip(",") + ",...")

summary = (
    "Summary of telemetry files (times shown in Asia/Shanghai UTC+8):\n\n"
    "metric.csv:\n"
    f" - total rows: {int(ms['rows_count'])}\n"
    f" - unique cmdb_id count: {int(ms['unique_cmdb_count'])}\n"
    f" - time range: {ms['min_timestamp_asia_shanghai']}  to  {ms['max_timestamp_asia_shanghai']}\n"
    f" - unique kpi_name count: {int(ms['unique_kpi_name_count'])}\n"
    f" - sample cmdb_ids: {compact_sample(ms['unique_cmdb_sample'])}\n"
    f" - sample kpi_names: {compact_sample(ms['unique_kpi_name_sample'])}\n"
    " - top 5 rows (CSV-like):\n"
    f"{metric_top5_csv}\n"
    "trace.csv:\n"
    f" - total rows: {int(ts['rows_count'])}\n"
    f" - unique cmdb_id count: {int(ts['unique_cmdb_count'])}\n"
    f" - time range: {ts['min_timestamp_asia_shanghai']}  to  {ts['max_timestamp_asia_shanghai']}\n"
    f" - unique trace_name count: {int(ts['unique_trace_name_count'])}\n"
    f" - sample cmdb_ids: {compact_sample(ts['unique_cmdb_sample'])}\n"
    f" - sample trace_names: {compact_sample(ts['unique_trace_name_sample'])}\n"
    " - top 5 rows (CSV-like):\n"
    f"{trace_top5_csv}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(  file  rows_count  unique_cmdb_count                                 unique_cmdb_sample min_timestamp_asia_shanghai max_timestamp_asia_shanghai  unique_kpi_name_count                             unique_kpi_name_sample
0  kpi       90791                 56  db_001,db_002,db_003,db_004,db_005,db_006,db_0...   2020-05-24T02:30:00+08:00   2020-05-24T03:30:00+08:00                    133  ACS,AIOS,AWS,CPU_Used_Pct,CPU_free_pct,Call_Pe...,           _dt_asia_shanghai   timestamp cmdb_id      kpi_name      value
0 2020-05-24 02:30:00+08:00  1590258600  db_001           ACS   0.010000
1 2020-05-24 02:30:00+08:00  1590258600  db_001          AIOS   0.000000
2 2020-05-24 02:30:00+08:00  1590258600  db_001           AWS   0.010000
3 2020-05-24 02:30:00+08:00  1590258600  db_001  CPU_Used_Pct   1.432018
4 2020-05-24 02:30:00+08:00  1590258600  db_001  CPU_free_pct  98.567982,     file  rows_count  unique_cmdb_count                                 unique_cmdb_sample min_timestamp_asia_shanghai max_timestamp_asia_shanghai  unique_trace_name_count                           unique_trace_name_sample
0  trace        6240                 10  docker_001,docker_002,docker_003,docker_004,do...   2020-05-24T03:00:00+08:00   2020-05-24T03:29:00+08:00                       76  trace.from_os_022.remoteprocess.duration_mean,...,           _dt_asia_shanghai   timestamp     cmdb_id                                     trace_name       value
0 2020-05-24 03:00:00+08:00  1590260400  docker_001  trace.from_os_022.remoteprocess.duration_mean    1.177968
1 2020-05-24 03:00:00+08:00  1590260400  docker_001   trace.from_os_022.remoteprocess.duration_p95    4.057400
2 2020-05-24 03:00:00+08:00  1590260400  docker_001     trace.from_os_022.remoteprocess.error_rate    0.000000
3 2020-05-24 03:00:00+08:00  1590260400  docker_001      trace.from_os_022.remoteprocess.row_count  253.000000
4 2020-05-24 03:00:00+08:00  1590260400  docker_001                   trace.self.csf.duration_mean    0.082796)```
```

In [ ]:
# In[2]:

# Reuse existing DataFrames: metric_df, trace_df
import pandas as pd

# 1) Metric.csv: unique cmdb_id count and prefix breakdown with samples
metric_unique_cmdb = pd.Series(metric_df['cmdb_id'].dropna().unique())
metric_unique_count = int(metric_unique_cmdb.size)

# categorize prefixes
def prefix_cat(name):
    if name.startswith('db_'):
        return 'db_'
    if name.startswith('os_'):
        return 'os_'
    if name.startswith('docker_'):
        return 'docker_'
    return 'other'

metric_prefix_groups = metric_unique_cmdb.to_frame(name='cmdb_id')
metric_prefix_groups['prefix'] = metric_prefix_groups['cmdb_id'].apply(prefix_cat)

# build summary rows: prefix, count, sample (up to 50)
prefix_summary_rows = []
for pref, grp in metric_prefix_groups.groupby('prefix'):
    samples = grp['cmdb_id'].sort_values().head(50).tolist()
    prefix_summary_rows.append({
        'prefix': pref,
        'count': int(grp.shape[0]),
        'sample_cmdb_ids': ",".join(samples)
    })
metric_prefix_summary_df = pd.DataFrame(prefix_summary_rows).sort_values('prefix').reset_index(drop=True)

# KPI names: full count and up to 200 sorted alphabetically
metric_kpi_unique = sorted(metric_df['kpi_name'].dropna().unique())
metric_kpi_count = len(metric_kpi_unique)
metric_kpi_sample = metric_kpi_unique[:200]
metric_kpi_summary_df = pd.DataFrame([{
    'unique_kpi_count': int(metric_kpi_count),
    'kpi_names_sample_count': len(metric_kpi_sample),
    'kpi_names_sample': ",".join(metric_kpi_sample)
}])

# 2) Trace.csv: unique cmdb_id count and list all unique cmdb_id values
trace_unique_cmdb = sorted(trace_df['cmdb_id'].dropna().unique())
trace_unique_cmdb_count = len(trace_unique_cmdb)
trace_cmdb_df = pd.DataFrame([{
    'unique_cmdb_count': int(trace_unique_cmdb_count),
    'cmdb_ids_all': ",".join(trace_unique_cmdb)
}])

# Trace names: full count and up to 200 sorted alphabetically
trace_name_unique = sorted(trace_df['trace_name'].dropna().unique())
trace_name_count = len(trace_name_unique)
trace_name_sample = trace_name_unique[:200]
trace_name_df = pd.DataFrame([{
    'unique_trace_name_count': int(trace_name_count),
    'trace_name_sample_count': len(trace_name_sample),
    'trace_name_sample': ",".join(trace_name_sample)
}])

# Display compact summaries (DataFrames)
metric_prefix_summary_df, metric_kpi_summary_df, trace_cmdb_df, trace_name_df

```
Out[2]:
```
```python
# Build a plain-English compact summary using existing summary DataFrames
# Reuse variables from prior execution: metric_prefix_summary_df, metric_kpi_summary_df, trace_cmdb_df, trace_name_df

def trunc(s, max_chars=400):
    if not isinstance(s, str):
        return str(s)
    return s if len(s) <= max_chars else s[:max_chars].rstrip(",") + ",..."

# Metric cmdb prefix breakdown
metric_prefix_df = metric_prefix_summary_df.copy()
total_metric_cmdb = int(metric_prefix_df['count'].sum())

prefix_lines = []
for _, r in metric_prefix_df.iterrows():
    pref = r['prefix']
    cnt = int(r['count'])
    sample = trunc(r['sample_cmdb_ids'], max_chars=200)
    prefix_lines.append(f"  - {pref}: {cnt} (sample: {sample})")

# Metric KPI summary
mk = metric_kpi_summary_df.iloc[0]
metric_kpi_count = int(mk['unique_kpi_count'])
metric_kpi_sample = trunc(mk['kpi_names_sample'], max_chars=800)

# Trace summaries
tc = trace_cmdb_df.iloc[0]
trace_cmdb_count = int(tc['unique_cmdb_count'])
trace_cmdb_list = trunc(tc['cmdb_ids_all'], max_chars=400)

tn = trace_name_df.iloc[0]
trace_name_count = int(tn['unique_trace_name_count'])
trace_name_sample = trunc(tn['trace_name_sample'], max_chars=800)

summary = (
    "Compact candidate components & KPI summary:\n\n"
    "1) metric.csv\n"
    f" - unique cmdb_id count: {total_metric_cmdb}\n"
    " - breakdown by prefix:\n"
    f"{chr(10).join(prefix_lines)}\n"
    f" - unique kpi_name count: {metric_kpi_count}\n"
    f" - kpi_name list (sorted, up to 200 items shown/truncated): {metric_kpi_sample}\n\n"
    "2) trace.csv\n"
    f" - unique cmdb_id count: {trace_cmdb_count}\n"
    f" - cmdb_id list: {trace_cmdb_list}\n"
    f" - unique trace_name count: {trace_name_count}\n"
    f" - trace_name list (sorted, up to 200 items shown/truncated): {trace_name_sample}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    prefix  count                                    sample_cmdb_ids
0      db_     13  db_001,db_002,db_003,db_004,db_005,db_006,db_0...
1  docker_      8  docker_001,docker_002,docker_003,docker_004,do...
2      os_     22  os_001,os_002,os_003,os_004,os_005,os_006,os_0...
3    other     13  osb_001,redis_001,redis_002,redis_003,redis_00...,    unique_kpi_count  kpi_names_sample_count                                   kpi_names_sample
0               133                     133  ACS,AIOS,AWS,Agent_ping,Asm_Free_Tb,Buffers_us...,    unique_cmdb_count                                       cmdb_ids_all
0                 10  docker_001,docker_002,docker_003,docker_004,do...,    unique_trace_name_count  trace_name_sample_count                                  trace_name_sample
0                       76                       76  trace.from_docker_001.remoteprocess.duration_m...)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# ensure timezone-aware datetime column in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if '_dt_asia_shanghai' not in metric_df.columns:
    metric_df['_dt_asia_shanghai'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, kpi_name) using all rows (no filtering)
thr = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(p05=lambda x: x.quantile(0.05), p95=lambda x: x.quantile(0.95))
)

# 2) Filter metric.csv to incident window inclusive
start = pd.Timestamp('2020-05-24 03:00:00', tz=tz)
end = pd.Timestamp('2020-05-24 03:30:00', tz=tz)
window_df = metric_df[(metric_df['_dt_asia_shanghai'] >= start) & (metric_df['_dt_asia_shanghai'] <= end)].copy()

# 3) Detect anomalies in the window using global thresholds
# merge thresholds into window rows
window_with_thr = window_df.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')

# define anomaly conditions
cond_high = window_with_thr['value'] > window_with_thr['p95']
cond_low = window_with_thr['value'] < window_with_thr['p05']
anom_mask = cond_high | cond_low

# build anomalies_df with requested columns and anomaly_type
anomalies_df = window_with_thr[anom_mask].copy()
if not anomalies_df.empty:
    anomalies_df['anomaly_type'] = np.where(cond_high[anom_mask] & cond_low[anom_mask], 'both',
                                            np.where(cond_high[anom_mask], 'high', 'low'))
else:
    anomalies_df['anomaly_type'] = pd.Series(dtype=str)

# keep only required columns (plus datetime for internal use)
anomalies_df = anomalies_df.loc[:, ['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'p05', 'anomaly_type', '_dt_asia_shanghai']]

# 4) Aggregate anomaly results per (cmdb_id, kpi_name)
# baseline groups are all (cmdb_id,kpi_name) pairs present in the window
pairs = window_with_thr.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    max_value_in_window=('value', 'max'),
    min_value_in_window=('value', 'min')
)

# attach global thresholds
pairs = pairs.merge(thr, on=['cmdb_id', 'kpi_name'], how='left')

# anomalies_count and earliest_anomaly_timestamp from anomalies_df
if not anomalies_df.empty:
    anom_counts = (
        anomalies_df
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(anomalies_count=('value', 'count'),
             earliest_anomaly_dt=('_dt_asia_shanghai', 'min'))
    )
else:
    anom_counts = pd.DataFrame(columns=['cmdb_id','kpi_name','anomalies_count','earliest_anomaly_dt'])

pairs = pairs.merge(anom_counts, on=['cmdb_id','kpi_name'], how='left')
pairs['anomalies_count'] = pairs['anomalies_count'].fillna(0).astype(int)

# format earliest_anomaly_timestamp as ISO string (Asia/Shanghai) or empty
pairs['earliest_anomaly_timestamp'] = pairs['earliest_anomaly_dt'].apply(
    lambda x: x.isoformat() if pd.notna(x) else None
)
pairs.drop(columns=['earliest_anomaly_dt'], inplace=True)

# compute severity per instructions, handling zero thresholds safely
def compute_severity(row):
    sev_high = 0.0
    sev_low = 0.0
    # high side
    if row['max_value_in_window'] > row['p95']:
        if row['p95'] == 0:
            sev_high = 9999.0
        else:
            sev_high = (row['max_value_in_window'] - row['p95']) / row['p95']
    # low side
    if row['min_value_in_window'] < row['p05']:
        if row['p05'] == 0:
            sev_low = 9999.0
        else:
            sev_low = (row['p05'] - row['min_value_in_window']) / row['p05']
    return max(sev_high, sev_low)

pairs['P95'] = pairs['p95']
pairs['P05'] = pairs['p05']
pairs['severity'] = pairs.apply(compute_severity, axis=1)

# select and order requested columns
agg = pairs.loc[:, ['cmdb_id', 'kpi_name', 'anomalies_count', 'earliest_anomaly_timestamp',
                    'max_value_in_window', 'min_value_in_window', 'P95', 'P05', 'severity']]

# 5) Return top 20 rows sorted by severity desc then anomalies_count desc
agg_top20 = agg.sort_values(['severity', 'anomalies_count'], ascending=[False, False]).head(20)

# counts
total_pairs_examined = int(pairs.shape[0])
pairs_with_anomaly = int((pairs['anomalies_count'] > 0).sum())

# Final displayed outputs: agg_top20, total_pairs_examined, pairs_with_anomaly
agg_top20, total_pairs_examined, pairs_with_anomaly

```
Out[3]:
```
```python
# Build a plain-English compact summary from previous results and display it.
# Reuse existing kernel variables: agg_top20, total_pairs_examined, pairs_with_anomaly

n_show = min(10, len(agg_top20))
rows = []
for i in range(n_show):
    r = agg_top20.iloc[i]
    sev = r['severity']
    # format severity, note sentinel
    sev_str = f"{sev:.3f}" if sev < 9999 else "9999 (sentinel for zero threshold)"
    rows.append(
        f"{i+1}. {r['cmdb_id']} | {r['kpi_name']} | anomalies: {int(r['anomalies_count'])} "
        f"| earliest: {r['earliest_anomaly_timestamp']} | severity: {sev_str}"
    )

summary = (
    f"Incident-window anomaly summary (2020-05-24 03:00 to 03:30 Asia/Shanghai):\n\n"
    f"- Total distinct (cmdb_id, kpi_name) pairs examined in window: {total_pairs_examined}\n"
    f"- Pairs with ≥1 anomaly in window: {pairs_with_anomaly}\n\n"
    "Top affected pairs by severity (top {0}, showing cmdb_id | kpi_name | anomalies | earliest_anomaly | severity):\n"
    .format(n_show)
    + ("\n".join(rows) if rows else "  (no anomalies detected in the top list)\n")
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id               kpi_name  anomalies_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window           P95     P05     severity
1643  os_020           Disk_io_util                2  2020-05-24T03:01:00+08:00             0.005609                  0.0      0.000000    0.00  9999.000000
47    db_002                   AIOS                1  2020-05-24T03:28:00+08:00             0.010000                  0.0      0.000000    0.00  9999.000000
126   db_003         Sess_Used_Temp                1  2020-05-24T03:01:00+08:00             1.000000                  0.0      0.000000    0.00  9999.000000
680   os_001             Disk_await                1  2020-05-24T03:14:00+08:00             3.200000                  0.0      0.000000    0.00  9999.000000
681   os_001           Disk_io_util                1  2020-05-24T03:21:00+08:00             0.005558                  0.0      0.000000    0.00  9999.000000
732   os_002           Disk_io_util                1  2020-05-24T03:15:00+08:00             0.005609                  0.0      0.000000    0.00  9999.000000
735   os_002             Disk_svctm                1  2020-05-24T03:10:00+08:00             1.230000                  0.0      0.000000    0.00  9999.000000
1087  os_009             Disk_svctm                1  2020-05-24T03:06:00+08:00             1.330000                  0.0      0.000000    0.00  9999.000000
1595  os_019             Disk_svctm                1  2020-05-24T03:23:00+08:00             9.690000                  0.0      0.000000    0.00  9999.000000
1746  os_022            Disk_rd_kbs                1  2020-05-24T03:00:00+08:00             0.000066                  0.0      0.000000    0.00  9999.000000
1766  os_022                Page_pi                1  2020-05-24T03:00:00+08:00             0.066721                  0.0      0.000000    0.00  9999.000000
1614  os_019                Page_pi                2  2020-05-24T03:17:00+08:00           165.461525                  0.0      1.257394    0.00   130.590797
1594  os_019            Disk_rd_kbs                2  2020-05-24T03:17:00+08:00             0.159590                  0.0      0.001219    0.00   129.961759
833   os_004           Disk_io_util                3  2020-05-24T03:18:00+08:00             0.500000                  0.0      0.004203    0.00   117.961938
70    db_002  Physical_Read_Per_Sec                1  2020-05-24T03:28:00+08:00            16.600000                  0.0      0.200000    0.00    82.000000
580   db_013           Redo_Per_Sec                4  2020-05-24T03:00:00+08:00         31251.100000                134.4    399.330000  139.33    77.258834
1592  os_019           Disk_io_util                2  2020-05-24T03:15:00+08:00             0.350172                  0.0      0.006459    0.00    53.214584
442   db_010           Redo_Per_Sec                4  2020-05-24T03:01:00+08:00         47570.900000                163.5   1026.730000  181.66    45.332434
534   db_012           Redo_Per_Sec                6  2020-05-24T03:00:00+08:00         28959.500000                421.7    898.470000  434.70    31.232017
13    db_001     Logic_Read_Per_Sec                3  2020-05-24T03:00:00+08:00        695311.800000                  6.0  21673.890000    6.00    31.080619, 2053, 980)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse anomalies_df, pairs if present in kernel
# anomalies_df should contain columns: cmdb_id, kpi_name, timestamp, value, p95, p05, anomaly_type, _dt_asia_shanghai
# pairs should contain per-(cmdb_id,kpi_name) severity; if not present we'll compute a fallback.

# Safety checks
try:
    anomalies_df
except NameError:
    anomalies_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p05','anomaly_type','_dt_asia_shanghai'])

# If pairs not available, try to use 'agg' (per-pair aggregated) or compute minimal severity map
severity_map_df = None
if 'pairs' in globals():
    severity_map_df = pairs.loc[:, ['cmdb_id', 'kpi_name', 'severity']].drop_duplicates()
elif 'agg' in globals():
    severity_map_df = agg.loc[:, ['cmdb_id', 'kpi_name', 'severity']].drop_duplicates()
else:
    severity_map_df = None

if anomalies_df.empty:
    # create empty aggregation
    anomalies_by_component_df = pd.DataFrame(columns=[
        'cmdb_id', 'anomalies_count', 'earliest_anomaly_timestamp',
        'unique_kpi_count', 'sample_kpis', 'max_severity'
    ])
    anomalies_by_component_df_top20 = anomalies_by_component_df.head(20)
    total_cmdb_with_anoms = 0
    total_anomaly_rows = 0
else:
    # aggregate anomaly counts, earliest timestamp, unique kpi count, sample kpis
    def sample_kpis_func(s):
        unique_kpis = sorted(pd.Index(s).unique())
        return ",".join(unique_kpis[:10])
    
    grp = (
        anomalies_df
        .groupby('cmdb_id', as_index=False)
        .agg(
            anomalies_count=('value', 'count'),
            earliest_anomaly_dt=('_dt_asia_shanghai', 'min'),
            unique_kpi_count=('kpi_name', lambda x: pd.Index(x).nunique()),
            sample_kpis=('kpi_name', sample_kpis_func)
        )
    )
    # format earliest timestamp
    grp['earliest_anomaly_timestamp'] = grp['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    grp.drop(columns=['earliest_anomaly_dt'], inplace=True)
    
    # compute max_severity per cmdb by merging severity_map_df for affected (cmdb_id,kpi_name)
    pairs_with_anoms = anomalies_df[['cmdb_id','kpi_name']].drop_duplicates()
    if severity_map_df is not None:
        # merge and compute max severity per cmdb_id
        merged = pairs_with_anoms.merge(severity_map_df, on=['cmdb_id','kpi_name'], how='left')
        # if some severities missing, fill with 0
        merged['severity'] = merged['severity'].fillna(0.0)
        max_sev = merged.groupby('cmdb_id', as_index=False).agg(max_severity=('severity','max'))
    else:
        # fallback: set severity 0 for all if we cannot find per-pair severities
        pairs_with_anoms['severity'] = 0.0
        max_sev = pairs_with_anoms.groupby('cmdb_id', as_index=False).agg(max_severity=('severity','max'))
    
    # join max_sev into grp
    anomalies_by_component_df = grp.merge(max_sev, on='cmdb_id', how='left')
    # ensure column order and types
    anomalies_by_component_df = anomalies_by_component_df.loc[:, [
        'cmdb_id', 'anomalies_count', 'earliest_anomaly_timestamp',
        'unique_kpi_count', 'sample_kpis', 'max_severity'
    ]]
    # sort by anomalies_count desc then max_severity desc
    anomalies_by_component_df = anomalies_by_component_df.sort_values(['anomalies_count','max_severity'], ascending=[False, False]).reset_index(drop=True)
    
    anomalies_by_component_df_top20 = anomalies_by_component_df.head(20)
    total_cmdb_with_anoms = int(anomalies_by_component_df.shape[0])
    total_anomaly_rows = int(anomalies_df.shape[0])

# Save full aggregation to kernel variable
# (already named anomalies_by_component_df)
anomalies_by_component_df, anomalies_by_component_df_top20, total_cmdb_with_anoms, total_anomaly_rows

```
Out[4]:
```
```python
# Build a plain-English compact summary from the aggregation results in kernel variables
# Reuse existing variables: anomalies_by_component_df_top20, total_cmdb_with_anoms, total_anomaly_rows

try:
    top = anomalies_by_component_df_top20
    total_cmdb = total_cmdb_with_anoms
    total_rows = total_anomaly_rows
except NameError:
    top = None
    total_cmdb = 0
    total_rows = 0

def fmt_sev(s):
    try:
        s = float(s)
        return f"{s:.3f}" if s < 9999 else "9999 (sentinel: zero threshold)"
    except:
        return str(s)

if top is None or top.empty:
    summary = "No anomalies found in the incident window."
else:
    # show up to top 10 components in the summary for compactness
    n = min(10, len(top))
    lines = []
    for i in range(n):
        r = top.iloc[i]
        lines.append(
            f"{i+1}. {r['cmdb_id']} | anomalies: {int(r['anomalies_count'])} | "
            f"unique_kpis: {int(r['unique_kpi_count'])} | earliest: {r['earliest_anomaly_timestamp']} | "
            f"max_severity: {fmt_sev(r['max_severity'])}"
        )
    summary = (
        f"Anomalies aggregated by component (incident window):\n\n"
        f"- Total distinct components with anomalies: {total_cmdb}\n"
        f"- Total anomaly rows: {total_rows}\n\n"
        f"Top components by anomaly count / severity (showing up to {n}):\n" +
        "\n".join(lines) +
        "\n\nNote: severity value 9999 indicates a zero P95/P05 threshold (sentinel for division-by-zero)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id  anomalies_count earliest_anomaly_timestamp  unique_kpi_count                                        sample_kpis  max_severity
0       os_019              108  2020-05-24T03:00:00+08:00                35  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
1       os_017               97  2020-05-24T03:00:00+08:00                32  CPU_idle_pct,CPU_system_time,CPU_user_time,CPU...      3.000000
2       os_020               96  2020-05-24T03:00:00+08:00                35  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
3       os_018               91  2020-05-24T03:00:00+08:00                31  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      9.045872
4       os_022               88  2020-05-24T03:00:00+08:00                32  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...   9999.000000
..         ...              ...                        ...               ...                                                ...           ...
50  docker_003                3  2020-05-24T03:21:00+08:00                 1                                 container_cpu_used      0.044521
51  docker_008                2  2020-05-24T03:09:00+08:00                 1                                 container_cpu_used      0.192982
52  docker_007                2  2020-05-24T03:03:00+08:00                 1                                 container_cpu_used      0.033592
53  docker_006                1  2020-05-24T03:02:00+08:00                 1                                 container_cpu_used      0.206897
54  docker_005                1  2020-05-24T03:06:00+08:00                 1                                 container_cpu_used      0.107143

[55 rows x 6 columns],    cmdb_id  anomalies_count earliest_anomaly_timestamp  unique_kpi_count                                        sample_kpis  max_severity
0   os_019              108  2020-05-24T03:00:00+08:00                35  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
1   os_017               97  2020-05-24T03:00:00+08:00                32  CPU_idle_pct,CPU_system_time,CPU_user_time,CPU...      3.000000
2   os_020               96  2020-05-24T03:00:00+08:00                35  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
3   os_018               91  2020-05-24T03:00:00+08:00                31  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      9.045872
4   os_022               88  2020-05-24T03:00:00+08:00                32  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...   9999.000000
5   os_009               84  2020-05-24T03:00:00+08:00                29  Buffers_used,CPU_idle_pct,CPU_system_time,CPU_...   9999.000000
6   os_005               78  2020-05-24T03:00:00+08:00                29  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...      1.000000
7   os_001               76  2020-05-24T03:00:00+08:00                26  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
8   os_021               74  2020-05-24T03:00:00+08:00                25  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...      0.892473
9   os_003               71  2020-05-24T03:00:00+08:00                30  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...      1.237347
10  os_002               69  2020-05-24T03:00:00+08:00                25  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...   9999.000000
11  db_009               69  2020-05-24T03:00:00+08:00                23  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DbT...      0.334770
12  os_014               68  2020-05-24T03:00:00+08:00                25  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      1.615802
13  db_002               67  2020-05-24T03:00:00+08:00                24  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...   9999.000000
14  db_007               65  2020-05-24T03:00:00+08:00                22  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...     19.000000
15  os_016               65  2020-05-24T03:01:00+08:00                26  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      3.092929
16  db_010               63  2020-05-24T03:00:00+08:00                24  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFP...     45.332434
17  os_004               61  2020-05-24T03:00:00+08:00                25  Buffers_used,CPU_idle_pct,CPU_iowait_time,CPU_...    117.961938
18  os_012               59  2020-05-24T03:00:00+08:00                27  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      2.961771
19  os_015               59  2020-05-24T03:02:00+08:00                25  CPU_idle_pct,CPU_iowait_time,CPU_system_time,C...      1.459869, 55, 2571)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df from kernel if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# ensure timezone-aware datetime column in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if '_dt_asia_shanghai' not in trace_df.columns:
    trace_df['_dt_asia_shanghai'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using all rows
trace_thr = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(P05=lambda x: x.quantile(0.05), P95=lambda x: x.quantile(0.95))
)

# 2) Filter trace.csv to the incident window inclusive
start = pd.Timestamp('2020-05-24 03:00:00', tz=tz)
end = pd.Timestamp('2020-05-24 03:30:00', tz=tz)
trace_window = trace_df[(trace_df['_dt_asia_shanghai'] >= start) & (trace_df['_dt_asia_shanghai'] <= end)].copy()

# total distinct (cmdb_id, trace_name) pairs examined in the window
total_pairs_examined = int(trace_window.groupby(['cmdb_id','trace_name']).ngroups)

# 3) Detect anomalies in the filtered rows by merging thresholds
trace_win_with_thr = trace_window.merge(trace_thr, on=['cmdb_id','trace_name'], how='left')

cond_high = trace_win_with_thr['value'] > trace_win_with_thr['P95']
cond_low = trace_win_with_thr['value'] < trace_win_with_thr['P05']
anom_mask = cond_high | cond_low

# 4) Save all anomaly rows into trace_anomalies_df with requested columns
if anom_mask.any():
    trace_anomalies_df = trace_win_with_thr[anom_mask].copy()
    trace_anomalies_df['anomaly_type'] = np.where(trace_anomalies_df['value'] > trace_anomalies_df['P95'], 'high',
                                                  np.where(trace_anomalies_df['value'] < trace_anomalies_df['P05'], 'low', 'both'))
    trace_anomalies_df = trace_anomalies_df.loc[:, ['cmdb_id', 'trace_name', 'timestamp', 'value', 'P95', 'P05', 'anomaly_type', '_dt_asia_shanghai']]
else:
    # empty DataFrame with required columns
    trace_anomalies_df = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'timestamp', 'value', 'P95', 'P05', 'anomaly_type', '_dt_asia_shanghai'])

# 5) Compute per-(cmdb_id,trace_name) severity using window values (max/min in window)
pairs_window = (
    trace_win_with_thr
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
    .merge(trace_thr, on=['cmdb_id','trace_name'], how='left')
)

def compute_severity_row(r):
    sev_high = 0.0
    sev_low = 0.0
    if r['max_value_in_window'] > r['P95']:
        if r['P95'] == 0:
            sev_high = 9999.0
        else:
            sev_high = (r['max_value_in_window'] - r['P95']) / r['P95']
    if r['min_value_in_window'] < r['P05']:
        if r['P05'] == 0:
            sev_low = 9999.0
        else:
            sev_low = (r['P05'] - r['min_value_in_window']) / r['P05']
    return max(sev_high, sev_low)

if not pairs_window.empty:
    pairs_window['severity'] = pairs_window.apply(compute_severity_row, axis=1)
else:
    pairs_window['severity'] = []

# number of distinct pairs with >=1 anomaly in the window
if not trace_anomalies_df.empty:
    pairs_with_anomaly = int(trace_anomalies_df[['cmdb_id','trace_name']].drop_duplicates().shape[0])
else:
    pairs_with_anomaly = 0

# For aggregation by cmdb_id, compute per-cmdb anomalies_count, earliest anomaly, unique_trace_count, sample_traces, and max_severity
if trace_anomalies_df.empty:
    trace_anomalies_by_comp_df = pd.DataFrame(columns=[
        'cmdb_id', 'anomalies_count', 'earliest_anomaly_timestamp',
        'unique_trace_count', 'sample_traces', 'max_severity'
    ])
else:
    # aggregate anomalies by cmdb_id
    def sample_traces_func(s):
        unique_traces = sorted(pd.Index(s).unique())
        return ",".join(unique_traces[:10])
    
    agg = (
        trace_anomalies_df
        .groupby('cmdb_id', as_index=False)
        .agg(
            anomalies_count=('value', 'count'),
            earliest_anomaly_dt=('_dt_asia_shanghai', 'min'),
            unique_trace_count=('trace_name', lambda x: pd.Index(x).nunique()),
            sample_traces=('trace_name', sample_traces_func)
        )
    )
    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_dt'].apply(lambda x: x.isoformat() if pd.notna(x) else None)
    agg.drop(columns=['earliest_anomaly_dt'], inplace=True)
    
    # compute max_severity per cmdb_id by joining per-pair severity for pairs that had anomalies
    affected_pairs = trace_anomalies_df[['cmdb_id','trace_name']].drop_duplicates()
    if not pairs_window.empty:
        sev_map = pairs_window[['cmdb_id','trace_name','severity']].drop_duplicates()
        merged = affected_pairs.merge(sev_map, on=['cmdb_id','trace_name'], how='left')
        merged['severity'] = merged['severity'].fillna(0.0)
        max_sev = merged.groupby('cmdb_id', as_index=False).agg(max_severity=('severity','max'))
    else:
        # fallback
        merged = affected_pairs.copy()
        merged['severity'] = 0.0
        max_sev = merged.groupby('cmdb_id', as_index=False).agg(max_severity=('severity','max'))
    
    trace_anomalies_by_comp_df = agg.merge(max_sev, on='cmdb_id', how='left')
    # ensure order and types
    trace_anomalies_by_comp_df = trace_anomalies_by_comp_df.loc[:, [
        'cmdb_id', 'anomalies_count', 'earliest_anomaly_timestamp',
        'unique_trace_count', 'sample_traces', 'max_severity'
    ]]
    # sort by anomalies_count desc then max_severity desc
    trace_anomalies_by_comp_df = trace_anomalies_by_comp_df.sort_values(['anomalies_count','max_severity'], ascending=[False, False]).reset_index(drop=True)

# save aggregation to kernel variable
# (trace_anomalies_df and trace_anomalies_by_comp_df are already named)

# 6) Prepare compact outputs: top-20 aggregated rows, total distinct pairs examined, number of pairs with ≥1 anomaly
trace_agg_top20 = trace_anomalies_by_comp_df.head(20)

trace_agg_top20, total_pairs_examined, pairs_with_anomaly

```
Out[5]:
```
```python
# Build a plain-English compact summary from trace anomaly aggregation results and display it.
# Reuse kernel variables: trace_agg_top20, total_pairs_examined, pairs_with_anomaly

top = trace_agg_top20.copy() if 'trace_agg_top20' in globals() else None
total_pairs = total_pairs_examined if 'total_pairs_examined' in globals() else None
pairs_with_anom = pairs_with_anomaly if 'pairs_with_anomaly' in globals() else None

def fmt_sev(s):
    try:
        s = float(s)
        return f"{s:.3f}" if s < 9999 else "9999 (sentinel: zero threshold)"
    except:
        return str(s)

if top is None or top.empty:
    summary = "No trace anomalies detected in the incident window."
else:
    n = min(10, len(top))
    lines = []
    for i in range(n):
        r = top.iloc[i]
        lines.append(
            f"{i+1}. {r['cmdb_id']} — anomalies: {int(r['anomalies_count'])}, "
            f"unique_traces: {int(r['unique_trace_count'])}, earliest: {r['earliest_anomaly_timestamp']}, "
            f"max_severity: {fmt_sev(r['max_severity'])}"
        )
    summary = (
        "Trace-based anomaly summary (2020-05-24 03:00 to 03:30 Asia/Shanghai):\n\n"
        f"- Total distinct (cmdb_id, trace_name) pairs examined in window: {total_pairs}\n"
        f"- Pairs with ≥1 anomaly in window: {pairs_with_anom}\n\n"
        "Top components by anomaly count (showing up to 10):\n" +
        "\n".join(lines) +
        "\n\nNote: severity value 9999 indicates a zero P95/P05 threshold (sentinel for division-by-zero)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  anomalies_count earliest_anomaly_timestamp  unique_trace_count                                      sample_traces  max_severity
0  docker_002               81  2020-05-24T03:00:00+08:00                  21  trace.from_os_022.remoteprocess.duration_mean,...      1.184736
1  docker_003               78  2020-05-24T03:00:00+08:00                  21  trace.from_os_021.remoteprocess.duration_mean,...      1.908501
2  docker_001               78  2020-05-24T03:00:00+08:00                  21  trace.from_os_022.remoteprocess.duration_mean,...      1.173830
3  docker_004               73  2020-05-24T03:00:00+08:00                  21  trace.from_os_021.remoteprocess.duration_mean,...      2.231635
4      os_022               50  2020-05-24T03:00:00+08:00                  14  trace.from_root.osb.duration_mean,trace.from_r...   9999.000000
5      os_021               46  2020-05-24T03:00:00+08:00                  12  trace.from_root.osb.duration_mean,trace.from_r...      1.203526
6  docker_005               41  2020-05-24T03:00:00+08:00                  11  trace.from_docker_003.remoteprocess.duration_m...      1.141755
7  docker_007               40  2020-05-24T03:00:00+08:00                  10  trace.from_docker_001.remoteprocess.duration_m...      0.963166
8  docker_008               38  2020-05-24T03:00:00+08:00                  11  trace.from_docker_001.remoteprocess.duration_m...      1.263693
9  docker_006               38  2020-05-24T03:00:00+08:00                  10  trace.from_docker_003.remoteprocess.duration_m...      1.126691, 208, 152)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse anomalies_df and trace_anomalies_df from kernel
# Ensure they exist
try:
    anomalies_df
except NameError:
    anomalies_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p05','anomaly_type','_dt_asia_shanghai'])
try:
    trace_anomalies_df
except NameError:
    trace_anomalies_df = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value','P95','P05','anomaly_type','_dt_asia_shanghai'])

# Normalize datetime column name and ensure tz-aware (should already be present)
# Find earliest datetime across both tables using their _dt_asia_shanghai columns as-is
dt_col_m = '_dt_asia_shanghai' if '_dt_asia_shanghai' in anomalies_df.columns else None
dt_col_t = '_dt_asia_shanghai' if '_dt_asia_shanghai' in trace_anomalies_df.columns else None

# Collect min datetimes safely
min_m = anomalies_df[dt_col_m].min() if (dt_col_m is not None and not anomalies_df.empty) else pd.NaT
min_t = trace_anomalies_df[dt_col_t].min() if (dt_col_t is not None and not trace_anomalies_df.empty) else pd.NaT

# earliest overall
earliest_dt = min(min_m, min_t)

# If no anomalies at all, create empty results
if pd.isna(earliest_dt):
    earliest_anomaly_records_df = pd.DataFrame(columns=[
        'origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','severity','_dt_asia_shanghai'
    ])
    earliest_anomaly_records_df_head20 = earliest_anomaly_records_df.head(20)
    earliest_ts_iso = None
    total_at_timestamp = 0
    components_involved = []
else:
    # Filter rows at that exact datetime (match tz-aware equality)
    # Metric anomalies: build standardized columns
    if not anomalies_df.empty:
        m = anomalies_df.copy()
        # rename p95/p05 -> P95/P05 for consistency
        if 'p95' in m.columns:
            m = m.rename(columns={'p95':'P95','p05':'P05'})
        m['origin'] = 'metric'
        m['name'] = m['kpi_name']
        # ensure numeric types
        m['P95'] = pd.to_numeric(m['P95'], errors='coerce')
        m['P05'] = pd.to_numeric(m['P05'], errors='coerce')
        m['value'] = pd.to_numeric(m['value'], errors='coerce')
    else:
        m = pd.DataFrame(columns=['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','_dt_asia_shanghai'])
    # Trace anomalies: standardize
    if not trace_anomalies_df.empty:
        t = trace_anomalies_df.copy()
        # trace_anomalies_df already uses P95/P05
        t['origin'] = 'trace'
        t['name'] = t['trace_name']
        t['P95'] = pd.to_numeric(t['P95'], errors='coerce')
        t['P05'] = pd.to_numeric(t['P05'], errors='coerce')
        t['value'] = pd.to_numeric(t['value'], errors='coerce')
    else:
        t = pd.DataFrame(columns=['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','_dt_asia_shanghai'])
    
    # Select rows where _dt_asia_shanghai == earliest_dt
    m_at = m[m['_dt_asia_shanghai'] == earliest_dt].copy() if not m.empty else m.iloc[0:0].copy()
    t_at = t[t['_dt_asia_shanghai'] == earliest_dt].copy() if not t.empty else t.iloc[0:0].copy()
    
    # Combine
    combined = pd.concat([
        m_at.loc[:, ['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','_dt_asia_shanghai']],
        t_at.loc[:, ['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','_dt_asia_shanghai']]
    ], ignore_index=True)
    
    # Compute severity per record using the same metric:
    # if anomaly_type == 'high' -> (value - P95)/P95 ; if P95==0 -> 9999
    # if anomaly_type == 'low'  -> (P05 - value)/P05 ; if P05==0 -> 9999
    def compute_sev_row(r):
        try:
            if r['anomaly_type'] == 'high':
                if pd.isna(r['P95']) or r['P95'] == 0:
                    return 9999.0
                return (r['value'] - r['P95']) / r['P95']
            elif r['anomaly_type'] == 'low':
                if pd.isna(r['P05']) or r['P05'] == 0:
                    return 9999.0
                return (r['P05'] - r['value']) / r['P05']
            else:
                # fallback: compute both sides and take max
                sev_high = 0.0
                sev_low = 0.0
                if not pd.isna(r.get('P95')) and r['value'] > r['P95']:
                    if r['P95'] == 0: sev_high = 9999.0
                    else: sev_high = (r['value'] - r['P95'])/r['P95']
                if not pd.isna(r.get('P05')) and r['value'] < r['P05']:
                    if r['P05'] == 0: sev_low = 9999.0
                    else: sev_low = (r['P05'] - r['value'])/r['P05']
                return max(sev_high, sev_low)
        except Exception:
            return np.nan

    if not combined.empty:
        combined['severity'] = combined.apply(compute_sev_row, axis=1)
    else:
        combined['severity'] = []

    # Final standardized columns and save full matches to kernel variable
    earliest_anomaly_records_df = combined.loc[:, ['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','severity','_dt_asia_shanghai']].copy()
    
    # Prepare compact output: limit 20 rows
    earliest_anomaly_records_df_head20 = earliest_anomaly_records_df.head(20)
    
    # Summary numbers
    earliest_ts_iso = earliest_dt.isoformat()
    total_at_timestamp = int(earliest_anomaly_records_df.shape[0])
    components_involved = sorted(earliest_anomaly_records_df['cmdb_id'].dropna().unique().tolist())[:50]

# Display compact results: head20, earliest timestamp, total count, components involved (limited)
earliest_anomaly_records_df_head20, earliest_ts_iso, total_at_timestamp, components_involved

```
Out[6]:
```
```python
# Build a concise plain-English summary string from existing kernel variables and display it.
# Reuse: earliest_anomaly_records_df_head20, earliest_ts_iso, total_at_timestamp, components_involved

try:
    table = earliest_anomaly_records_df_head20.loc[:, ['origin','cmdb_id','name','timestamp','value','P95','P05','anomaly_type','severity']]
    table_csv = table.to_csv(index=False)
except Exception:
    table_csv = "(no anomaly records to show)"

components_list = ",".join(components_involved[:50]) if components_involved else ""

summary = (
    f"Earliest anomaly timestamp (Asia/Shanghai): {earliest_ts_iso}\n"
    f"Total anomaly records at this timestamp: {total_at_timestamp}\n"
    f"Distinct components involved ({len(components_involved)} shown up to 50): {components_list}\n\n"
    "Anomaly records at that timestamp (up to 20 rows, CSV-like):\n"
    f"{table_csv}"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    origin cmdb_id                 name   timestamp         value           P95          P05 anomaly_type   severity         _dt_asia_shanghai
0   metric  db_001   Logic_Read_Per_Sec  1590260400  80305.800000  21673.890000     6.000000         high   2.705186 2020-05-24 03:00:00+08:00
1   metric  db_002         CPU_Used_Pct  1590260400      2.447626      2.014959     0.906471         high   0.214727 2020-05-24 03:00:00+08:00
2   metric  db_002         CPU_free_pct  1590260400     97.552374     99.134249    98.535826          low   0.009981 2020-05-24 03:00:00+08:00
3   metric  db_002  DFParaWrite_Per_Sec  1590260400      3.800000      0.548000     0.130000         high   5.934307 2020-05-24 03:00:00+08:00
4   metric  db_002         Sess_Connect  1590260400    276.000000    279.000000   277.000000          low   0.003610 2020-05-24 03:00:00+08:00
5   metric  db_005   Logic_Read_Per_Sec  1590260400  40078.600000   3155.170000  1019.440000         high  11.702517 2020-05-24 03:00:00+08:00
6   metric  db_005        Proc_Used_Pct  1590260400      2.790000      2.780500     2.780000         high   0.003417 2020-05-24 03:00:00+08:00
7   metric  db_005   Proc_User_Used_Pct  1590260400      2.790000      2.780500     2.780000         high   0.003417 2020-05-24 03:00:00+08:00
8   metric  db_006      SeqRead_Per_Sec  1590260400      0.820000      0.780500     0.000000         high   0.050609 2020-05-24 03:00:00+08:00
9   metric  db_007                 AIOS  1590260400      0.010000      0.000500     0.000000         high  19.000000 2020-05-24 03:00:00+08:00
10  metric  db_009       LFSync_Per_Sec  1590260400      0.620000      0.610500     0.280000         high   0.015561 2020-05-24 03:00:00+08:00
11  metric  db_010         CPU_Used_Pct  1590260400      2.084044      2.078031     1.791786         high   0.002894 2020-05-24 03:00:00+08:00
12  metric  db_010         CPU_free_pct  1590260400     97.915956     98.215375    97.932278          low   0.000167 2020-05-24 03:00:00+08:00
13  metric  db_011         CPU_free_pct  1590260400     98.672542     99.113132    98.692505          low   0.000202 2020-05-24 03:00:00+08:00
14  metric  db_011  LFParaWrite_Per_Sec  1590260400      0.050000      0.170000     0.059500          low   0.159664 2020-05-24 03:00:00+08:00
15  metric  db_012         Call_Per_Sec  1590260400     33.900000     33.315000    30.795000         high   0.017560 2020-05-24 03:00:00+08:00
16  metric  db_012         Exec_Per_Sec  1590260400     30.800000     24.900000    21.600000         high   0.236948 2020-05-24 03:00:00+08:00
17  metric  db_012  LFParaWrite_Per_Sec  1590260400      0.200000      0.190500     0.090000         high   0.049869 2020-05-24 03:00:00+08:00
18  metric  db_012         Redo_Per_Sec  1590260400  28959.500000    898.470000   434.700000         high  31.232017 2020-05-24 03:00:00+08:00
19  metric  db_012          TPS_Per_Sec  1590260400      2.500000      2.105000     1.600000         high   0.187648 2020-05-24 03:00:00+08:00, '2020-05-24T03:00:00+08:00', 218, ['db_001', 'db_002', 'db_005', 'db_006', 'db_007', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022', 'osb_001', 'redis_001', 'redis_002', 'redis_003', 'redis_004', 'redis_005', 'redis_006', 'redis_007', 'redis_009', 'redis_010', 'redis_011', 'redis_012'])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse anomalies_df from kernel
try:
    anomalies_df
except NameError:
    anomalies_df = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p05','anomaly_type','_dt_asia_shanghai'])

# Ensure datetime column exists and tz is Asia/Shanghai
tz = pytz.timezone('Asia/Shanghai')
if '_dt_asia_shanghai' not in anomalies_df.columns and 'timestamp' in anomalies_df.columns:
    anomalies_df['_dt_asia_shanghai'] = pd.to_datetime(anomalies_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Normalize threshold column names if needed (p95/p05 -> P95/P05)
df = anomalies_df.copy()
if 'p95' in df.columns:
    df = df.rename(columns={'p95': 'P95', 'p05': 'P05'})

# Filter to DB services only
db_df = df[df['cmdb_id'].str.startswith('db_', na=False)].copy()

# Prepare empty result if no DB anomalies
if db_df.empty:
    db_anomaly_summary_df = pd.DataFrame(columns=[
        'cmdb_id','anomalies_count','earliest_anomaly_timestamp','unique_kpi_count',
        'sample_kpis','max_severity','top_kpi','top_kpi_severity','top_kpi_timestamp','top_kpi_value'
    ])
    db_anomaly_summary_df_top10 = db_anomaly_summary_df.head(10)
    total_db_components_examined = 0
    db_components_with_anomalies = 0
else:
    # Compute per (cmdb_id, kpi_name) pair metrics from anomaly rows
    pair_grp = (
        db_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            pair_anomalies_count=('value','count'),
            earliest_anomaly_dt=('_dt_asia_shanghai','min'),
            max_value_in_window=('value','max'),
            min_value_in_window=('value','min'),
            P95=('P95','first'),
            P05=('P05','first')
        )
    )
    # compute severity per pair
    def pair_severity(r):
        sev_high = 0.0
        sev_low = 0.0
        # high side
        if r['max_value_in_window'] > r['P95']:
            if pd.isna(r['P95']) or r['P95'] == 0:
                sev_high = 9999.0
            else:
                sev_high = (r['max_value_in_window'] - r['P95']) / r['P95']
        # low side
        if r['min_value_in_window'] < r['P05']:
            if pd.isna(r['P05']) or r['P05'] == 0:
                sev_low = 9999.0
            else:
                sev_low = (r['P05'] - r['min_value_in_window']) / r['P05']
        return max(sev_high, sev_low)
    pair_grp['severity'] = pair_grp.apply(pair_severity, axis=1)
    pair_grp['earliest_anomaly_timestamp'] = pair_grp['earliest_anomaly_dt'].dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notna(x) else None)
    pair_grp.drop(columns=['earliest_anomaly_dt'], inplace=True)
    
    # For each db, compute aggregated metrics
    def sample_kpis_func(s):
        uniques = sorted(pd.Index(s).unique())
        return ",".join(uniques[:10])
    
    db_agg = (
        db_df
        .groupby('cmdb_id', as_index=False)
        .agg(
            anomalies_count=('value','count'),
            earliest_anomaly_dt=('_dt_asia_shanghai','min'),
            unique_kpi_count=('kpi_name', lambda x: pd.Index(x).nunique()),
            sample_kpis=('kpi_name', sample_kpis_func)
        )
    )
    db_agg['earliest_anomaly_timestamp'] = db_agg['earliest_anomaly_dt'].dt.tz_convert(tz).apply(lambda x: x.isoformat() if pd.notna(x) else None)
    db_agg.drop(columns=['earliest_anomaly_dt'], inplace=True)
    
    # compute max_severity per db from pair_grp
    max_sev = pair_grp.groupby('cmdb_id', as_index=False).agg(max_severity=('severity','max'))
    db_agg = db_agg.merge(max_sev, on='cmdb_id', how='left')
    
    # determine top_kpi per db: highest severity, tie-breaker earliest anomaly timestamp, then highest pair_anomalies_count
    # merge pair metadata to choose top kpi
    pair_for_selection = pair_grp[['cmdb_id','kpi_name','severity','pair_anomalies_count','earliest_anomaly_timestamp','max_value_in_window']]
    # for tie-breakers, sort appropriately
    pair_for_selection = pair_for_selection.sort_values(['cmdb_id','severity','earliest_anomaly_timestamp','pair_anomalies_count'],
                                                        ascending=[True, False, True, False])
    top_kpi = pair_for_selection.groupby('cmdb_id', as_index=False).first().rename(columns={
        'kpi_name':'top_kpi',
        'severity':'top_kpi_severity',
        'earliest_anomaly_timestamp':'top_kpi_timestamp',
        'max_value_in_window':'top_kpi_value'
    })
    db_anomaly_summary_df = db_agg.merge(top_kpi[['cmdb_id','top_kpi','top_kpi_severity','top_kpi_timestamp','top_kpi_value']], on='cmdb_id', how='left')
    
    # final columns order and types
    db_anomaly_summary_df = db_anomaly_summary_df.loc[:, [
        'cmdb_id','anomalies_count','earliest_anomaly_timestamp','unique_kpi_count',
        'sample_kpis','max_severity','top_kpi','top_kpi_severity','top_kpi_timestamp','top_kpi_value'
    ]]
    
    # sort by top_kpi_severity desc then anomalies_count desc
    db_anomaly_summary_df = db_anomaly_summary_df.sort_values(['top_kpi_severity','anomalies_count'], ascending=[False, False]).reset_index(drop=True)
    
    # prepare top 10 for display
    db_anomaly_summary_df_top10 = db_anomaly_summary_df.head(10)
    
    # summary numbers
    total_db_components_examined = int(db_df['cmdb_id'].nunique())
    db_components_with_anomalies = int((db_anomaly_summary_df['anomalies_count'] > 0).sum())

# Display compact outputs: top 10 rows and two summary numbers (use variable names per rules)
db_anomaly_summary_df, db_anomaly_summary_df_top10, total_db_components_examined, db_components_with_anomalies

```
Out[7]:
```
Summary of DB (db_*) anomalies (incident window), full table saved as db_anomaly_summary_df.

Top 10 DB components (sorted by top_kpi_severity desc, then anomalies_count desc):
1. db_002 — anomalies_count: 67; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 24; top_kpi: AIOS (severity=9999) at 2020-05-24T03:28:00+08 (value=0.01); max_severity: 9999  
2. db_003 — anomalies_count: 57; earliest: 2020-05-24T03:01:00+08; unique_kpi_count: 21; top_kpi: Sess_Used_Temp (severity=9999) at 2020-05-24T03:01:00+08 (value=1.00); max_severity: 9999  
3. db_013 — anomalies_count: 52; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 22; top_kpi: Redo_Per_Sec (severity=77.258834) at 2020-05-24T03:00:00+08 (value=31251.10); max_severity: 77.258834  
4. db_010 — anomalies_count: 63; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 24; top_kpi: Redo_Per_Sec (severity=45.332434) at 2020-05-24T03:01:00+08 (value=47570.90); max_severity: 45.332434  
5. db_012 — anomalies_count: 43; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 15; top_kpi: Redo_Per_Sec (severity=31.232017) at 2020-05-24T03:00:00+08 (value=28959.50); max_severity: 31.232017  
6. db_001 — anomalies_count: 48; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 21; top_kpi: Logic_Read_Per_Sec (severity=31.080619) at 2020-05-24T03:00:00+08 (value=695311.80); max_severity: 31.080619  
7. db_011 — anomalies_count: 48; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 22; top_kpi: Redo_Per_Sec (severity=22.142791) at 2020-05-24T03:01:00+08 (value=30113.40); max_severity: 22.142791  
8. db_007 — anomalies_count: 65; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 22; top_kpi: AIOS (severity=19.000000) at 2020-05-24T03:00:00+08 (value=0.01); max_severity: 19.000000  
9. db_005 — anomalies_count: 48; earliest: 2020-05-24T03:00:00+08; unique_kpi_count: 20; top_kpi: Logic_Read_Per_Sec (severity=11.702517) at 2020-05-24T03:00:00+08 (value=40078.60); max_severity: 11.702517  
10. db_008 — anomalies_count: 47; earliest: 2020-05-24T03:01:00+08; unique_kpi_count: 18; top_kpi: DFParaWrite_Per_Sec (severity=1.000000) at 2020-05-24T03:02:00+08 (value=2.41); max_severity: 1.000000

Summary counts:
- Total distinct db_* components examined (in anomalies_df): 13  
- Number of db_* components with ≥1 anomaly: 13

Note: severity = relative deviation beyond P95 or below P05; a severity value of 9999 indicates a zero threshold (sentinel for division-by-zero). Full results are saved in db_anomaly_summary_df for follow-up.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  anomalies_count earliest_anomaly_timestamp  unique_kpi_count                                        sample_kpis  max_severity              top_kpi  top_kpi_severity          top_kpi_timestamp  top_kpi_value
0   db_002               67  2020-05-24T03:00:00+08:00                24  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...   9999.000000                 AIOS       9999.000000  2020-05-24T03:28:00+08:00          0.010
1   db_003               57  2020-05-24T03:01:00+08:00                21  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...   9999.000000       Sess_Used_Temp       9999.000000  2020-05-24T03:01:00+08:00          1.000
2   db_013               52  2020-05-24T03:00:00+08:00                22  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     77.258834         Redo_Per_Sec         77.258834  2020-05-24T03:00:00+08:00      31251.100
3   db_010               63  2020-05-24T03:00:00+08:00                24  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFP...     45.332434         Redo_Per_Sec         45.332434  2020-05-24T03:01:00+08:00      47570.900
4   db_012               43  2020-05-24T03:00:00+08:00                15  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFP...     31.232017         Redo_Per_Sec         31.232017  2020-05-24T03:00:00+08:00      28959.500
5   db_001               48  2020-05-24T03:00:00+08:00                21  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     31.080619   Logic_Read_Per_Sec         31.080619  2020-05-24T03:00:00+08:00     695311.800
6   db_011               48  2020-05-24T03:00:00+08:00                22  CPU_Used_Pct,CPU_free_pct,DFParaWrite_Per_Sec,...     22.142791         Redo_Per_Sec         22.142791  2020-05-24T03:01:00+08:00      30113.400
7   db_007               65  2020-05-24T03:00:00+08:00                22  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...     19.000000                 AIOS         19.000000  2020-05-24T03:00:00+08:00          0.010
8   db_005               48  2020-05-24T03:00:00+08:00                20  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     11.702517   Logic_Read_Per_Sec         11.702517  2020-05-24T03:00:00+08:00      40078.600
9   db_008               47  2020-05-24T03:01:00+08:00                18  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...      1.000000  DFParaWrite_Per_Sec          1.000000  2020-05-24T03:02:00+08:00          2.410
10  db_004               42  2020-05-24T03:01:00+08:00                22  ACS,AIOS,CPU_Used_Pct,CPU_free_pct,DFParaWrite...      1.000000                  ACS          1.000000  2020-05-24T03:08:00+08:00          0.030
11  db_006               36  2020-05-24T03:00:00+08:00                15  ACS,CPU_Used_Pct,CPU_free_pct,DFParaWrite_Per_...      0.390244          Session_pct          0.390244  2020-05-24T03:09:00+08:00          0.057
12  db_009               69  2020-05-24T03:00:00+08:00                23  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DbT...      0.334770         Exec_Per_Sec          0.334770  2020-05-24T03:01:00+08:00        233.100,   cmdb_id  anomalies_count earliest_anomaly_timestamp  unique_kpi_count                                        sample_kpis  max_severity              top_kpi  top_kpi_severity          top_kpi_timestamp  top_kpi_value
0  db_002               67  2020-05-24T03:00:00+08:00                24  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...   9999.000000                 AIOS       9999.000000  2020-05-24T03:28:00+08:00           0.01
1  db_003               57  2020-05-24T03:01:00+08:00                21  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...   9999.000000       Sess_Used_Temp       9999.000000  2020-05-24T03:01:00+08:00           1.00
2  db_013               52  2020-05-24T03:00:00+08:00                22  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     77.258834         Redo_Per_Sec         77.258834  2020-05-24T03:00:00+08:00       31251.10
3  db_010               63  2020-05-24T03:00:00+08:00                24  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFP...     45.332434         Redo_Per_Sec         45.332434  2020-05-24T03:01:00+08:00       47570.90
4  db_012               43  2020-05-24T03:00:00+08:00                15  ACS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFP...     31.232017         Redo_Per_Sec         31.232017  2020-05-24T03:00:00+08:00       28959.50
5  db_001               48  2020-05-24T03:00:00+08:00                21  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     31.080619   Logic_Read_Per_Sec         31.080619  2020-05-24T03:00:00+08:00      695311.80
6  db_011               48  2020-05-24T03:00:00+08:00                22  CPU_Used_Pct,CPU_free_pct,DFParaWrite_Per_Sec,...     22.142791         Redo_Per_Sec         22.142791  2020-05-24T03:01:00+08:00       30113.40
7  db_007               65  2020-05-24T03:00:00+08:00                22  AIOS,CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DF...     19.000000                 AIOS         19.000000  2020-05-24T03:00:00+08:00           0.01
8  db_005               48  2020-05-24T03:00:00+08:00                20  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...     11.702517   Logic_Read_Per_Sec         11.702517  2020-05-24T03:00:00+08:00       40078.60
9  db_008               47  2020-05-24T03:01:00+08:00                18  CPU_Used_Pct,CPU_free_pct,Call_Per_Sec,DFParaW...      1.000000  DFParaWrite_Per_Sec          1.000000  2020-05-24T03:02:00+08:00           2.41, 13, 13)```
```